<a href="https://colab.research.google.com/github/PavithraSivakumar-12/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PavithraSivakumar-12/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

print("✅ Hugging Face token loaded successfully!")

✅ Hugging Face token loaded successfully!


In [2]:
rel = "hf://datasets/FlyRank/internship-warehouse"

In [3]:
con.sql(f"""
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01


## 1. My rule and its reason codes

### Rule
Pages with high Google Search impressions but very low clicks should be reviewed because they may have low click-through performance.

### Score
The baseline score increases when:
- Google Search impressions are high.
- Google Search clicks are low.

### Reason Code
LOW_CTR_HIGH_IMPRESSIONS

### Action
Review the page title and meta description to improve click-through rate.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [5]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    AVG(gsc_impressions) AS avg_impressions,
    AVG(gsc_clicks) AS avg_clicks
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE gsc_data_available IS TRUE
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,avg_impressions,avg_clicks
0,28970051,60.8739,0.222327


## 2. Build the ranked queue (writes the CSV)

Pages are ranked using a simple baseline score.

The score gives higher priority to pages with:
- High Google Search impressions
- Low Google Search clicks

Reason Code:
LOW_CTR_HIGH_IMPRESSIONS

Action:
Review title and meta description.

In [9]:
import os

os.makedirs("work/outputs", exist_ok=True)

baseline = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,

    (gsc_impressions - (gsc_clicks * 10)) AS baseline_score,

    'LOW_CTR_HIGH_IMPRESSIONS' AS reason_code,
    'Review title and meta description' AS action

FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')

WHERE
    gsc_data_available IS TRUE

ORDER BY baseline_score DESC

LIMIT 100
""").df()

baseline.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

baseline.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,baseline_score,reason_code,action
0,2026-06-11,client_e547b89c05043229,content_963de14b1f58978f,245826,890,236926,LOW_CTR_HIGH_IMPRESSIONS,Review title and meta description
1,2025-12-08,client_23a62021009f63c4,content_469512b149928d3e,156642,32,156322,LOW_CTR_HIGH_IMPRESSIONS,Review title and meta description
2,2026-01-24,client_e547b89c05043229,content_1e921148b5fee86a,83308,102,82288,LOW_CTR_HIGH_IMPRESSIONS,Review title and meta description
3,2026-04-20,client_3f0ce4d44fe94f3d,content_468d0aa0891d425d,82317,129,81027,LOW_CTR_HIGH_IMPRESSIONS,Review title and meta description
4,2026-01-23,client_e547b89c05043229,content_1e921148b5fee86a,81086,110,79986,LOW_CTR_HIGH_IMPRESSIONS,Review title and meta description
5,2026-01-21,client_e547b89c05043229,content_fff3da2dddec56d0,80716,194,78776,LOW_CTR_HIGH_IMPRESSIONS,Review title and meta description
6,2026-01-21,client_e547b89c05043229,content_a4ffb46244c68d04,82179,474,77439,LOW_CTR_HIGH_IMPRESSIONS,Review title and meta description
7,2026-01-24,client_e547b89c05043229,content_3bf07141cf1aa743,76144,33,75814,LOW_CTR_HIGH_IMPRESSIONS,Review title and meta description
8,2026-02-15,client_73cda7b4e4f265ea,content_fec55986a1868d62,74688,0,74688,LOW_CTR_HIGH_IMPRESSIONS,Review title and meta description
9,2026-02-20,client_73cda7b4e4f265ea,content_8e1334d6356668e3,62844,0,62844,LOW_CTR_HIGH_IMPRESSIONS,Review title and meta description


## 3. Top-20 review

The highest-ranked pages were reviewed using the baseline rule.

For each page:
- Action: Review title and meta description.
- Reason Code: LOW_CTR_HIGH_IMPRESSIONS.
- Confidence: Medium, because the score only uses impressions and clicks.
- What would make it wrong: High impressions alone do not always indicate a problem. Some pages may already perform well for their search intent or keyword competition.

In [10]:
top20 = baseline.head(20).copy()

top20["confidence_note"] = "Medium"
top20["what_would_make_it_wrong"] = (
    "High impressions alone may not indicate poor performance."
)

top20

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,baseline_score,reason_code,action,confidence_note,what_would_make_it_wrong
0,2026-06-11,client_e547b89c05043229,content_963de14b1f58978f,245826,890,236926,LOW_CTR_HIGH_IMPRESSIONS,Review title and meta description,Medium,High impressions alone may not indicate poor p...
1,2025-12-08,client_23a62021009f63c4,content_469512b149928d3e,156642,32,156322,LOW_CTR_HIGH_IMPRESSIONS,Review title and meta description,Medium,High impressions alone may not indicate poor p...
2,2026-01-24,client_e547b89c05043229,content_1e921148b5fee86a,83308,102,82288,LOW_CTR_HIGH_IMPRESSIONS,Review title and meta description,Medium,High impressions alone may not indicate poor p...
3,2026-04-20,client_3f0ce4d44fe94f3d,content_468d0aa0891d425d,82317,129,81027,LOW_CTR_HIGH_IMPRESSIONS,Review title and meta description,Medium,High impressions alone may not indicate poor p...
4,2026-01-23,client_e547b89c05043229,content_1e921148b5fee86a,81086,110,79986,LOW_CTR_HIGH_IMPRESSIONS,Review title and meta description,Medium,High impressions alone may not indicate poor p...
5,2026-01-21,client_e547b89c05043229,content_fff3da2dddec56d0,80716,194,78776,LOW_CTR_HIGH_IMPRESSIONS,Review title and meta description,Medium,High impressions alone may not indicate poor p...
6,2026-01-21,client_e547b89c05043229,content_a4ffb46244c68d04,82179,474,77439,LOW_CTR_HIGH_IMPRESSIONS,Review title and meta description,Medium,High impressions alone may not indicate poor p...
7,2026-01-24,client_e547b89c05043229,content_3bf07141cf1aa743,76144,33,75814,LOW_CTR_HIGH_IMPRESSIONS,Review title and meta description,Medium,High impressions alone may not indicate poor p...
8,2026-02-15,client_73cda7b4e4f265ea,content_fec55986a1868d62,74688,0,74688,LOW_CTR_HIGH_IMPRESSIONS,Review title and meta description,Medium,High impressions alone may not indicate poor p...
9,2026-02-20,client_73cda7b4e4f265ea,content_8e1334d6356668e3,62844,0,62844,LOW_CTR_HIGH_IMPRESSIONS,Review title and meta description,Medium,High impressions alone may not indicate poor p...


## 4. Weak picks + leakage check

## Weak Picks and Leakage Check

Some highly ranked pages may not actually require action because the baseline rule only considers impressions and clicks.

This baseline does not use future information, product flags, or target labels. The score is calculated only from observed search metrics available at the time of prediction.

In [11]:
print("Leakage Check")

print("✓ No future-window variables used")
print("✓ No label-derived variables used")
print("✓ Only observed GSC metrics used")
print("✓ Baseline score uses impressions and clicks only")

Leakage Check
✓ No future-window variables used
✓ No label-derived variables used
✓ Only observed GSC metrics used
✓ Baseline score uses impressions and clicks only


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.